In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_customer(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/customer.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
customer = load_customer(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

def_codes = ["13", "31", "23", "29", "30", "18", "17"]

# 1. extract country code and initial filters
customer_filtered = (
    customer[["C_CUSTKEY", "C_ACCTBAL", "C_PHONE"]]
    .assign(CNTRYCODE=customer.C_PHONE.str.slice(0, 2))
)
customer_filtered = customer_filtered[
    (customer_filtered.C_ACCTBAL > 0.0)
    & (customer_filtered.CNTRYCODE.isin(def_codes))
]

# 2. filter above-average balances
avg_balance = customer_filtered.C_ACCTBAL.mean()
customer_filtered = customer_filtered[customer_filtered.C_ACCTBAL > avg_balance]

# 3. anti-join: drop customers with any orders
customer_filtered = customer_filtered[~customer_filtered.C_CUSTKEY.isin(orders.O_CUSTKEY)]

# 4. single groupby for both aggregations
total = (
    customer_filtered
    .groupby("CNTRYCODE")
    .agg({"C_CUSTKEY": "count", "C_ACCTBAL": "sum"})
    .reset_index()
    .rename(columns={"C_CUSTKEY": "NUMCUST", "C_ACCTBAL": "TOTACCTBAL"})
    .sort_values("CNTRYCODE")
)